# MVP unificado - Redes neuronales confiables

Este notebook es el punto de entrada narrativo del MVP unificado. No sustituye al notebook pesado ya validado (`01_mvp_dani_professional.ipynb`); por ahora organiza la historia del proyecto y consume artefactos ya generados por el pipeline validado.

La implementacion canonica expuesta aqui usa `src.trustworthy_credit`, que durante la migracion reexporta el core validado de Dani sin modificarlo.

## 0. Configuracion global

Centralizamos rutas, estilo y utilidades de reporting. Esta celda no entrena modelos ni recalcula resultados.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / "results"
TABLES_DIR = RESULTS_DIR / "tables"
FIGURES_DIR = RESULTS_DIR / "figures"

from src.trustworthy_credit.plots import ParetoPlotter, UncertaintyPlotter
from src.trustworthy_credit.reporting import (
    DatasetOverviewReporter,
    ResultsTableFormatter,
    UncertaintyNarrativeReporter,
)

pd.set_option("display.max_columns", 80)
plt.rcParams["figure.figsize"] = (9, 5)

print(f"Project root: {PROJECT_ROOT}")
print(f"Tables dir:   {TABLES_DIR}")
print(f"Figures dir:  {FIGURES_DIR}")

## 1. Contrato de datos e inventario

El dataset usado es Home Credit Default Risk. La variable objetivo es `TARGET` y la variable sensible auditada es `CODE_GENDER`. El contrato formal vive en `src.trustworthy_credit.data_contract`.

In [ ]:
from src.trustworthy_credit.data_contract import build_default_home_credit_contract

contract = build_default_home_credit_contract()
print(contract)

## 2. EDA inicial

El EDA compacto sigue la idea narrativa del notebook de Javi: desbalanceo de `TARGET`, distribucion por genero y calidad de `EXT_SOURCE`. Si el CSV local no esta disponible, esta seccion queda documentada sin bloquear el resto del notebook.

In [ ]:
raw_path = PROJECT_ROOT / "data" / "raw" / "application_train.csv"
if raw_path.exists():
    raw = pd.read_csv(
        raw_path,
        usecols=[
            "TARGET",
            "CODE_GENDER",
            "EXT_SOURCE_1",
            "EXT_SOURCE_2",
            "EXT_SOURCE_3",
        ],
    )
    raw = raw[raw["CODE_GENDER"].isin(["M", "F"])]
    overview = DatasetOverviewReporter()
    display(overview.target_summary(raw))
    display(overview.gender_summary(raw))
    display(overview.ext_source_missingness(raw))
else:
    print(f"Raw dataset not found at {raw_path}.")
    print("Run the professional training notebook or place Kaggle data locally to populate this section.")

## 3. Preprocesamiento y split

La base canonica es el pipeline POO de Dani, expuesto por `src.trustworthy_credit.preprocessing` y `src.trustworthy_credit.splitting`. El punto critico ya corregido es preservar `EXT_NULL_COUNT` crudo como feature de auditoria semantica.

In [ ]:
from src.trustworthy_credit.preprocessing import HomeCreditMVPPreprocessingPipeline
from src.trustworthy_credit.splitting import HomeCreditTrainValTestSplitter

print(HomeCreditMVPPreprocessingPipeline)
print(HomeCreditTrainValTestSplitter)

## 4. Arquitectura custom

La arquitectura combina un MLP tabular con capas custom de interpretabilidad financiera. En la migracion unificada se exponen desde `src.trustworthy_credit.layers` y `src.trustworthy_credit.models`.

In [ ]:
from src.trustworthy_credit.layers import FinancialRatiosLayer, TrainableGammaLayer
from src.trustworthy_credit.models import CustomMLPModelBuilder, FairCustomModelBuilder

print(FinancialRatiosLayer)
print(TrainableGammaLayer)
print(CustomMLPModelBuilder)
print(FairCustomModelBuilder)

## 5. FAIR loss

La penalizacion FAIR busca reducir dependencia entre prediccion y genero. La implementacion validada usa una penalizacion cuadratica sobre la correlacion para empujar la dependencia hacia cero, no hacia una correlacion negativa.

In [ ]:
from src.trustworthy_credit.layers import FairnessPenalty
from src.trustworthy_credit.metrics import FairnessMetricCalculator

print(FairnessPenalty)
print(FairnessMetricCalculator)

## 6. AutoML

La busqueda de arquitectura y el barrido de `lambda_fair` se mantienen en la capa de tuning validada. El notebook maestro solo orquesta y presenta resultados.

In [ ]:
from src.trustworthy_credit.tuning import FairKerasTunerRunner, FairLambdaSweepTrainer

print(FairKerasTunerRunner)
print(FairLambdaSweepTrainer)

## 7. Pareto lambda_fair

La Pareto oficial del MVP Dani no se sustituye. Aqui solo se presenta desde artefactos ya generados, usando la utilidad comun `ParetoPlotter`.

In [ ]:
pareto_path = TABLES_DIR / "pareto_results.csv"
if pareto_path.exists():
    pareto_df = pd.read_csv(pareto_path)
    display(ResultsTableFormatter().pareto_summary_table(pareto_df))
    fig, ax = ParetoPlotter().plot_auc_vs_fairness(pareto_df)
    plt.show()
else:
    print(f"Pareto artifact not found at {pareto_path}.")

## 8. Evaluacion test base vs FAIR

La tabla final resume el trade-off: perdida moderada de AUC a cambio de una reduccion fuerte de dependencia con genero.

In [ ]:
metrics_path = TABLES_DIR / "test_results_base_vs_fair.csv"
if metrics_path.exists():
    metrics_df = pd.read_csv(metrics_path)
    display(ResultsTableFormatter().base_vs_fair_table(metrics_df))
else:
    print(f"Test metrics artifact not found at {metrics_path}.")

## 9. Incertidumbre M2

La incertidumbre principal del MVP es el modelo M2: aprende el error absoluto esperado del clasificador M1. Esta via esta alineada con el enunciado y se mantiene como resultado principal.

In [ ]:
uncertainty_path = TABLES_DIR / "uncertainty_test.csv"
if uncertainty_path.exists():
    uncertainty_df = pd.read_csv(uncertainty_path)
    reporter = UncertaintyNarrativeReporter()
    display(reporter.summary_by_target(uncertainty_df))
    display(reporter.summary_by_ext_null_count(uncertainty_df))
else:
    print(f"Uncertainty artifact not found at {uncertainty_path}.")

## 10. Figuras obligatorias

Las figuras obligatorias se generan desde artefactos validados: Pareto, comparativa base vs FAIR, distribucion de incertidumbre por clase y relacion entre incertidumbre y `EXT_NULL_COUNT`.

In [ ]:
if uncertainty_path.exists():
    uncertainty_df = pd.read_csv(uncertainty_path)
    plotter = UncertaintyPlotter()
    fig, ax = plotter.plot_by_target(uncertainty_df)
    plt.show()
    fig, ax = plotter.plot_by_ext_null_count(uncertainty_df)
    plt.show()
else:
    print("Uncertainty plots skipped because uncertainty_test.csv is not available.")

## 11. Conclusiones y limitaciones

- El MVP validado combina precision, justicia estadistica e incertidumbre.
- La Pareto principal se mantiene como resultado Dani defendible.
- M2 duda mas donde el clasificador espera cometer mas error, y permite analizar la calidad de `EXT_SOURCE` mediante `EXT_NULL_COUNT`.
- La limitacion metodologica principal es que el MVP de incertidumbre usa validation como base de errores; la evolucion natural para nota alta es OOF uncertainty.
- MC Dropout, Pareto multi-semilla y features relacionales quedan como extras controlados, no como sustitutos del MVP.